# Fig. S3 — MSD statistics

Computes the parquet files read by `msd_new.py`, which draws the figure.

In [ ]:
from vertex_lite.custom import load_data_v3 as load_data
import pandas as pd
import os
import gc
import pyprojroot.here as here

In [ ]:
def filter_data(local=False, cluster=False):
    if cluster:
        rpath = here('results/resultados_cluster')
    else:
        rpath = here('results/resultados_random')

    dfs = []
    for n in [0.1, 0.5]:
         dfs.append(load_data(
             f'{rpath}/noise_{n}/pct_*/sim_*/cells.parquet',
             columns=['time','cell_id', 'neighbor_of_mutant', 'alive','mutant','centroid_x','centroid_y']
         ))
    
    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print("No data found")
        return pd.DataFrame()

    df = df[df['alive'] == 1]

    if local:
        df = df[df['neighbor_of_mutant'] == True]
    else:
        df = df.drop(columns=['neighbor_of_mutant'])
    
    return df

In [ ]:
def compute_msd_t0(df, max_quantile=0.9995):
    """Squared displacement of every cell relative to its position at t = 0."""
    df = df.copy()
    grouped = df.groupby(['mesh_type', 'pct_mutant', 'sim_id', 'cell_id'], sort=False)
    dx = df['centroid_x'] - grouped['centroid_x'].transform('first')
    dy = df['centroid_y'] - grouped['centroid_y'].transform('first')

    df['squared displacement'] = dx**2 + dy**2

    cutoff = df['squared displacement'].quantile(max_quantile)
    df = df[df['squared displacement'] <= cutoff].copy()
    return df

In [ ]:
def calculate_msd_stats(df):
    groupby_cols = ['time', 'mesh_type', 'pct_mutant']
    msd_stats = (
        df.groupby(groupby_cols)['squared displacement']
        .agg(msd_global='mean', msd_global_std='std', msd_global_sem='sem')
        .reset_index()
    )
    return msd_stats

In [ ]:
# The SI figure uses the mutant-neighbourhood (local) cells and the
# cumulative displacement from t = 0 ("total" timescale).
local = True
timescale = 'total'
savepath = here(f'results/analysis/msd/msd_stats/{timescale}')
os.makedirs(savepath, exist_ok=True)

for cluster in [True, False]:
    df = filter_data(local=local, cluster=cluster)
    msd = compute_msd_t0(df)
    stats = calculate_msd_stats(msd)
    del msd

    stats.to_parquet(f"{savepath}/msd_stats_{'local' if local else 'global'}_{'cluster' if cluster else 'random'}.parquet")

    del stats
    gc.collect()